In [8]:
%%writefile HPC_1B.cpp
#include
#include
#include
#include
using namespace std;

const int MAX = 100;

vector graph[MAX];
bool visited[MAX];

// Parallel DFS function
void parallelDFS(int start) {
    stack s;
    s.push(start);

    while (!s.empty()) {
        int curr;

        // Critical section for safe stack access
        #pragma omp critical
        {
            if (!s.empty()) {
                curr = s.top();
                s.pop();
            }
        }

        // Check and mark visited
        if (!visited[curr]) {
            visited[curr] = true;

            // Print node
            #pragma omp critical
            cout << curr << " ";

            // Explore neighbors in parallel
            #pragma omp parallel for
            for (int i = 0; i < graph[curr].size(); i++) {
                int adj = graph[curr][i];

                if (!visited[adj]) {
                    #pragma omp critical
                    s.push(adj);
                }
            }
        }
    }
}

int main() {
    int n, m, start;

    cout << "Enter number of nodes, edges, start node: ";
    cin >> n >> m >> start;

    cout << "Enter edges (u v):\n";
    for (int i = 0; i < m; i++) {
        int u, v;
        cin >> u >> v;

        graph[u].push_back(v);
        graph[v].push_back(u); // undirected graph
    }

    // Initialize visited array
    #pragma omp parallel for
    for (int i = 0; i < n; i++) {
        visited[i] = false;
    }

    cout << "Parallel DFS Traversal: ";
    parallelDFS(start);

    return 0;
}


Overwriting HPC_1B.cpp


In [9]:
!g++ -fopenmp HPC_1B.cpp -o HPC_1B

HPC_1B.cpp:1:10: error: #include expects "FILENAME" or <FILENAME>
    1 | #include
      |          ^
HPC_1B.cpp:2:10: error: #include expects "FILENAME" or <FILENAME>
    2 | #include
      |          ^
HPC_1B.cpp:3:10: error: #include expects "FILENAME" or <FILENAME>
    3 | #include
      |          ^
HPC_1B.cpp:4:10: error: #include expects "FILENAME" or <FILENAME>
    4 | #include
      |          ^
HPC_1B.cpp:9:1: error: ‘vector’ does not name a type
    9 | vector graph[MAX];
      | ^~~~~~
HPC_1B.cpp: In function ‘void parallelDFS(int)’:
HPC_1B.cpp:14:5: error: ‘stack’ was not declared in this scope
   14 |     stack s;
      |     ^~~~~
HPC_1B.cpp:1:1: note: ‘std::stack’ is defined in header ‘<stack>’; did you forget to ‘#include <stack>’?
  +++ |+#include <stack>
    1 | #include
HPC_1B.cpp:15:5: error: ‘s’ was not declared in this scope
   15 |     s.push(start);
      |     ^
HPC_1B.cpp:35:13: error: ‘cout’ was not declared in this scope
   35 |             cout << curr << 

In [10]:
!./HPC_1B

Enter number of nodes, edges, start node: 5 5 1
Enter edges (u v):
1 2
2 3
2 4
3 4
4 5
Parallel DFS Traversal: 1 2 3 4 5 